# Colab Training: Anomaly Detection

**How to use:**
1. Go to **Runtime > Change runtime type > TPU** (free tier works)
2. Upload your dataset to `/content/raw_dataset/` with class folders (`fight/`, `theft/`, `intrusion/`, `normal/`), or keep it on Google Drive and update `RAW_DATASET_DIR` below
3. **Run All** (Ctrl+F9) — that's it

The 2 shell commands below handle everything: clone, install, config, training, eval, and export.

In [ ]:
import os, json
from pathlib import Path

# ──────────────── EDIT THESE ────────────────
GITHUB_REPO_URL  = "https://github.com/AakashBhat1/collab_training_anomaly_detection.git"
GITHUB_BRANCH    = "main"
KAGGLE_DATASET   = "webadvisor/real-time-anomaly-detection-in-cctv-surveillance"
KAGGLE_USERNAME  = "aakash1233445"
REPO_DIR         = "/content/collab_training_anomaly_detection"
# ─────────────────────────────────────────────

# Mount Google Drive (checkpoint backup + resume across sessions)
from google.colab import drive
drive.mount("/content/drive")

# Load Kaggle API key from Colab Secrets (tries common name variations)
from google.colab import userdata
kaggle_key = ""
for secret_name in ["Kagle_key", "kagle_key", "KAGLE_KEY", "kaggle_key", "KAGGLE_KEY"]:
    try:
        kaggle_key = userdata.get(secret_name)
        if kaggle_key:
            print(f"Found Kaggle key in Colab Secret: '{secret_name}'")
            break
    except Exception:
        continue

if kaggle_key and KAGGLE_USERNAME:
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"]      = kaggle_key
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / "kaggle.json").write_text(
        json.dumps({"username": KAGGLE_USERNAME, "key": kaggle_key})
    )
    (kaggle_dir / "kaggle.json").chmod(0o600)
    print(f"Kaggle credentials ready for: {KAGGLE_USERNAME}")
else:
    print("ERROR: No Kaggle key found in Colab Secrets.")
    print("Add a secret with Name: Kagle_key  Value: <your kaggle api key>")

In [ ]:
# Step 1: Clone repo + install deps + update config + pull Kaggle dataset
import subprocess
from pathlib import Path

if not Path(f"{REPO_DIR}/.git").exists():
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch",
                     GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

!bash {REPO_DIR}/scripts/colab_setup.sh \
    {GITHUB_REPO_URL} \
    --branch {GITHUB_BRANCH} \
    --repo-dir {REPO_DIR} \
    --kaggle-dataset {KAGGLE_DATASET} \
    --kaggle-clean

In [ ]:
# Step 2: Train — split dataset, train model, evaluate, export to ONNX/OpenVINO
!bash {REPO_DIR}/scripts/colab_run_training.sh {REPO_DIR}